#### Libraries

In [3]:
import numpy as np
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
from scipy.stats import norm

#### Parameters

In [4]:
alpha = 0.95
k = norm.pdf(norm.ppf(alpha)) / (1 - alpha)
print(k) 

2.0627128075074257


#### Sets

In [5]:
df = pd.read_csv("selected_monthly_return.csv", encoding="utf-8-sig")
tickers = df["Ticker"].tolist()
returns = df.drop(columns=["PERMNO", "Ticker", "average_monthly_return"])
mu = df["average_monthly_return"].to_numpy(dtype=float)

#### Covariance Matrix

In [6]:
cov_df = pd.read_csv("covariance_matrix.csv", skiprows=1, encoding="utf-8-sig")

Sigma = cov_df.drop(columns=["PERMNO", "Ticker"]).to_numpy(dtype=float)

eigvals, eigvecs = np.linalg.eigh(Sigma)
A = np.sqrt(np.clip(eigvals, 0, None))[:, None] * eigvecs.T

### Model
$$
\begin{align*}
\min_{w}\quad &-\sum_{i\in I}\hat \mu_i^{\rm M}w_i+k_\alpha \lVert Aw\rVert_2. \\
s.t. \quad &\sum_{i \in I} w_i = 1 \\
&0.1u_i \leq w_i \leq 0.2 u_i & \forall i \in I \\
&\sum_{i \in I} \hat\mu^M_i w_I \geq b \\
&w_i \geq 0 & \forall i \in I \\
&u_i \in \{0, 1\} & \forall i \in I
\end{align*}
$$

In [7]:
# model
m = gp.Model("portfolio")

Set parameter Username
Set parameter LicenseID to value 2871508


Academic license - for non-commercial use only - expires 2027-10-05


#### Decision Variables

In [8]:
# weights
w = m.addMVar(len(tickers), vtype=GRB.CONTINUOUS, name="w")
# indicator variable for investing in a specific stock
z = m.addMVar(len(tickers), vtype=GRB.BINARY, name="u")
# standard deviation of portfolio (helper variable)
s = m.addVar(vtype=GRB.CONTINUOUS, name="s")

#### Constraints

In [9]:
# full investment of capital
m.addConstr(w.sum() == 1, name="full_investment")
# min and max weight
m.addConstr(w <= 0.2 * z, name="max_investment")
m.addConstr(0.01 * z <= w, name = "min_investment")

<MConstr (37,) *awaiting model update*>

In [10]:
# average returns at least as good as s&p
benchmark = pd.read_csv("SP500_monthly_202001_202603.csv", encoding="utf-8-sig")
benchmark["Month"] = pd.to_datetime(benchmark["Month"])
b = benchmark.loc[benchmark["Month"].between("2021-01-01", "2025-12-31"), "MonthlyReturn"].mean()
m.addConstr(w @ mu >= b, name="minimum_return")

<MConstr () *awaiting model update*>

In [11]:
# helper constraint for the standard deviation to be added to objective function
y = m.addMVar(37, lb=-GRB.INFINITY, name="y") # helper variable for finding s 
m.addConstr(y == A @ w)
m.addGenConstrNorm(s, y, 2.0) # constraint to find s

<gurobi.GenConstr *Awaiting Model Update*>

#### Objective Function

In [12]:
m.setObjective(-1 * w @ mu + k * s, gp.GRB.MINIMIZE)

#### Results

In [13]:
m.optimize()

Gurobi Optimizer version 13.0.3 build v13.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) Ultra 9 185H, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 22 logical processors, using up to 22 threads

Optimize a model with 113 rows, 112 columns and 1628 nonzeros (Min)
Model fingerprint: 0x6f3c4632
Model has 38 linear objective coefficients
Model has 1 simple general constraint
  1 NORM
Variable types: 75 continuous, 37 integer (37 binary)
Coefficient statistics:
  Matrix range     [1e-06, 1e+00]
  Objective range  [2e-03, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e-02, 1e+00]

Presolve time: 0.01s
Presolved: 114 rows, 149 columns, 1660 nonzeros
Presolved model has 37 quadratic constraint(s)
Variable types: 112 continuous, 37 integer (37 binary)

Root relaxation: objective -3.969804e-02, 43 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Dep

In [14]:
portfolio = pd.DataFrame({"Ticker": tickers, "Optimized Weight": w.X})
print(portfolio.to_string(index=False, formatters={"Optimized Weight": "{:.3%}".format}))

Ticker Optimized Weight
  COHR           0.000%
    GE           0.000%
   FNV           0.000%
    UI           0.000%
  BABA           3.801%
  LITE           0.000%
  CVNA           0.000%
   CVS           6.812%
   NET           0.000%
  PLTR           0.000%
  SOFI           0.000%
   NEM           0.000%
   CAH          19.981%
   GLW           0.000%
   HWM           0.000%
  KLAC           0.000%
  LRCX           0.000%
    MU           0.000%
  INTC           0.000%
   AMD           0.000%
   WDC           0.000%
     B           0.000%
  IDXX           0.000%
   AEM           0.000%
  ESLT          20.000%
  CIEN           0.000%
   APH           0.000%
   FIX           7.819%
    MT           0.000%
   CLS           0.000%
  INSM           4.157%
    DB           8.114%
   STX           0.000%
   KGC           0.000%
   NRG           0.000%
   WPM          14.054%
  AVGO          15.262%


In [15]:
portfolio_return = mu @ w.X

print(f"Portfolio returns: {portfolio_return:.6%}")
print(f"Portfolio monthly standard deviation: {s.X:.6%}")
print(f"Portfolio monthly 95% CVaR: {m.ObjVal:.6%}")

Portfolio returns: 3.004309%
Portfolio monthly standard deviation: 4.331404%
Portfolio monthly 95% CVaR: 5.930133%


In [16]:
print(f"Historical benchmark mean monthly return b: {b:.6%}") # 2021-2025
print(f"Excess return: {(portfolio_return - b):.6%}")

Historical benchmark mean monthly return b: 1.099027%
Excess return: 1.905282%


### Actual Portfolio Return vs. Actual S&P 500 Return in Jan 2026

In [ ]:
# Out-of-sample evaluation: January 2026
sp500_jan = benchmark.loc[
    benchmark["Month"].dt.to_period("M") == "2026-01",
    "MonthlyReturn"
].iloc[0]

actual_jan_return = df["26-Jan"].to_numpy(dtype=float) @ w.X

actual_excess_return = actual_jan_return - sp500_jan

print(f"Actual portfolio return in January 2026: {actual_jan_return:.6%}")
print(f"Actual S&P 500 return in January 2026: {sp500_jan:.6%}")
print(f"Actual excess return in January 2026: {actual_excess_return:.6%}")

Actual portfolio return in January 2026: 8.060061%
Actual S&P 500 return in January 2026: 1.366446%
Actual portfolio excess return in January 2026: 6.693616%
